# 中心链：沿序列的接触参与概率

横轴为零起始残基位置，纵轴为该残基至少参与一个有效分子内接触的概率。
每条链、每帧先记录“有/无接触”，再做帧内链平均和有效帧平均。
不是平均接触伙伴数，也不是成对概率矩阵逐行求和。阴影为时间块SD，不是置信区间。不展示热图。


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "project.py").is_file())
sys.path.insert(0, str(ROOT))
import project
import result_files as files

SELECTED = None
OUTPUT = ROOT / "02_calculate/03_contacts/01_central_chain/01_intramolecular/output"
SIMULATIONS = project.select_simulations(ROOT / "simulations.yaml", SELECTED)
RESULTS = {s["id"]: files.completed(OUTPUT / s["id"]) for s in SIMULATIONS}
SOURCES = {name: files.read_json(path / "metadata.json") for name, path in RESULTS.items()}
COLORS = {name: plt.get_cmap("tab10")(i % 10) for i, name in enumerate(RESULTS)}
FIGURES = OUTPUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
plt.rcParams.update(
    {
        "font.size": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "legend.frameon": False,
    }
)


def export(fig, name):
    for key, path in RESULTS.items():
        files.completed(path)
        if files.read_json(path / "metadata.json") != SOURCES[key]:
            raise ValueError("Results changed; restart notebook")
    for ext in ("png", "pdf", "svg"):
        fig.savefig(FIGURES / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    files.write_json(FIGURES / f"{name}.sources.json", SOURCES)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5), constrained_layout=True)
for name, path in RESULTS.items():
    data = pd.read_csv(path / "residue_probability/summary.csv").sort_values("residue_index")
    x, y, sd = (
        data.residue_index.to_numpy(),
        data.fraction_mean_frames.to_numpy(),
        data.fraction_std_blocks.to_numpy(),
    )
    ax.plot(x, y, color=COLORS[name], label=name, lw=1.3)
    ax.fill_between(x, np.maximum(0, y - sd), np.minimum(1, y + sd), color=COLORS[name], alpha=0.15)
ax.set(
    xlabel="Residue index (0-based)",
    ylabel="Probability of at least one intrachain contact",
    ylim=(0, 1),
    title="Frame-equal mean ± block SD",
)
ax.legend()
export(fig, "residue_contact_probability")
plt.show()
